# Attention Information Content

Information-theoretic analysis of attention: entropy profiles,
mutual information, concentration, and information flow rates.

## Why This Matters

Attention information content reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_information_content import (
    attention_entropy_profile, attention_mutual_information,
    attention_concentration, information_flow_rate,
    information_content_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Entropy Profile

Sharp vs diffuse attention across heads.

In [ ]:
for layer in range(4):
    result = attention_entropy_profile(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_H={result['mean_entropy']:.4f}")
    for h in result['per_head']:
        flag = ' [SHARP]' if h['is_sharp'] else ''
        bar = '█' * int(h['mean_entropy'] * 5)
        print(f"    H{h['head']}: H={h['mean_entropy']:.4f}, norm={h['mean_normalized_entropy']:.4f} {bar}{flag}")

## Mutual Information

How input-dependent is each head's attention?

In [ ]:
for layer in range(4):
    result = attention_mutual_information(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_MI={result['mean_mi']:.4f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: MI={h['mutual_information']:.4f}, "
              f"H(marginal)={h['marginal_entropy']:.4f}")

## Attention Concentration

How much mass is concentrated in top positions?

In [ ]:
for layer in range(4):
    for head in range(4):
        r = attention_concentration(model, tokens, layer, head)
        flag = ' [CONC]' if r['is_concentrated'] else ''
        print(f"  L{layer}H{head}: top1={r['mean_top1_mass']:.4f}{flag}")

## Information Flow Rate

Estimated information transferred through attention.

In [ ]:
for layer in range(4):
    result = information_flow_rate(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_flow={result['mean_flow']:.4f}")
    for h in result['per_head']:
        bar = '█' * int(h['mean_information_flow'] * 20)
        print(f"    H{h['head']}: flow={h['mean_information_flow']:.4f} {bar}")

## Information Content Summary

In [ ]:
result = information_content_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f}, MI={p['mean_mi']:.4f}")